In [1]:
from qiskit import QuantumCircuit , transpile
from qiskit.primitives import BackendSamplerV2 
import matplotlib.pyplot as plt
import pylatexenc
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
import numpy as np

In [4]:
#Oracle pada kuantum
def quantum_oracle(target):
    oracle = QuantumCircuit(len(target) , name = 'Oracle')


    #Menerapkan X-gate untuk setiap menemukan |0>
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)
    
    #Menerapkan multi controlled Z-gate untuk semua qubit
    for i, bit in enumerate(reversed(target)):
        if bit == '0':
            oracle.x(i)

    #Membuat black box
    oracle.to_gate()
    return oracle

# Menampilkan Oracle
quantum_oracle('10').draw()

#Inisialisasi
target = '10'
circuit = QuantumCircuit(len(target))
circuit.h(list(range(len(target)))) # Superposisi semua qubit
circuit.append(quantum_oracle(target), list(range(len(target))))

#Simulasi 
simulator = AerSimulator(method = 'statevector')

compiled = transpile(circuit, simulator)
result = simulator.run(compiled).result()
statevector = result.get_statevector()

counts = result.get_counts()
print(counts)

#Mengambil aplitudo setiap keadaan
statevector = result.get_statevector()

#Menampilkan amplitudo setiap keadaan 
for i, amp in enumerate(np.asarray(statevector)):
    print(f"Amplitude |{bin(i)[2:].zfill(len(target))}> : {amp:.2f}")
circuit.draw()      

def quantum_reflection(target):
    ref = QuantumCircuit(len(target), name='Reflection')

    #X-gate untuk semua qubit
    ref.x(list(range(len(target))))

    #Menerapkan multi controlled Z-gate untuk semua qubit 
    ref.h(len(target)- 1)
    ref.mcx(list(range(len(target)- 1)), len(target)- 1)
    ref.h(len(target)- 1)

    #X-gate untuk semua qubit 
    ref.x(list(range(len(target))))

    #Membuat black box
    ref.to_gate()
    return ref

#Menampilkan quantum reflection
quantum_reflection('10').draw()

#Inisialisasi
target = '10'
n = len(target) # Banyak qubit
N = 2**n # Banyak data

circuit = QuantumCircuit(n)
circuit.h(list(range(n))) # Superposisi semua qubit

#Menerapkan oracle dan reflection
circuit.append(quantum_oracle(target), list(range(n)))
circuit.h(list(range(n)))
circuit.append(quantum_reflection(target), list(range(n)))
circuit.h(list(range(n)))

#Simulasi
simulator = AerSimulator.get_backend("statevector_simulator")
job = transpile(circuit, simulator)
result = simulator.run(job).result()

#Mengambil amplitudo setiap keadaan 
statevector = result.get_statevector()

#Menampilkan amplitudo setiap keadaan
for i, amp in enumerate(np.asarray(statevector)):
    print(f"Amplitude |{bin(i)[2:].zfill(len(target))}> : {amp:.2f}")

#Menampilkan 
circuit.draw()

#Inisialisasi
target = '01011'
n = len(target)
N = 2**n # Banyaknya data acak

grover_circ = QuantumCircuit(n)
grover_circ.h(range(n))

#Menerapkan iterasi optimal
optimal = int((np.pi / 4) * np.sqrt(N))
for _ in range(optimal):
    grover_circ.append(quantum_oracle(target), list(range(n)))
    grover_circ.h(list(range(n)))
    grover_circ.append(quantum_reflection(target), list(range(n)))
    grover_circ.h(list(range(n)))

#Ukur semua keadaan
grover_circ.measure_all()
print(f"{target} ditemukan!")
print(f"Membutuhkan {optimal} iterasi supaya optimal")

#Tampilkan circuit
grover_circ.draw('mpl')
sampler = BackendSamplerV2(backend=aer)
job = sampler.run(grover_circ)
result = job.result()

labels = list(result.quasi_dists[0].keys())
values = list(result.quasi_dists[0].values())

plt.bar(labels, values, color='orange')
plt.show()


QiskitError: 'No statevector for experiment "None"'